# 两阶段训练快速入门

简化的两阶段训练 Notebook，使用实际数据。

In [1]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import matplotlib.pyplot as plt
import cv2

print(f"PyTorch: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.4.0+cu121


In [2]:
# 工具函数
def percentile_norm(x, pmin=0.1, pmax=99.9, eps=1e-8):
    lo = np.percentile(x, pmin)
    hi = np.percentile(x, pmax)
    return np.clip((x - lo) / max(hi - lo, eps), 0.0, 1.0)

def load_image(path):
    arr = np.array(Image.open(path)).astype(np.float32)
    return arr.mean(axis=2) if arr.ndim == 3 else arr

def save_image(path, x):
    x = np.clip(x / max(x.max(), 1e-8), 0.0, 1.0)
    Image.fromarray((x * 65535.0).astype(np.uint16)).save(path)

In [3]:
# 数据集
class TwoStageDataset(Dataset):
    def __init__(self, data_path, input_y=128, input_x=128, insert_xy=16, n_samples=100):
        self.img_files = sorted(glob.glob(os.path.join(data_path, "*.tif")))
        self.input_y = input_y
        self.input_x = input_x
        self.insert_xy = insert_xy
        self.n_samples = n_samples
        print(f"Found {len(self.img_files)} images in {data_path}")
    
    def __len__(self):
        return self.n_samples
    
    def __getitem__(self, idx):
        file_idx = np.random.randint(0, len(self.img_files))
        img = load_image(self.img_files[file_idx])
        
        if img.shape != (self.input_y, self.input_x):
            img = cv2.resize(img, (self.input_x, self.input_y))
        
        # 添加噪声
        noisy = img + np.random.normal(0, 0.1, img.shape).astype(np.float32)
        noisy = np.clip(noisy, 0, 1)
        
        # Padding
        noisy_padded = np.pad(noisy, ((self.insert_xy, self.insert_xy),
                                       (self.insert_xy, self.insert_xy)),
                              mode='constant')
        
        return (
            torch.from_numpy(noisy_padded[None]),
            torch.from_numpy(img[None])
        )

# 测试
dataset = TwoStageDataset("datasets/Microtubule/train_data", n_samples=10)
input_batch, gt_batch = dataset[0]
print(f"Input: {input_batch.shape}, GT: {gt_batch.shape}")

Found 0 images in datasets/Microtubule/train_data


ValueError: high <= 0

In [ ]:
# 模型定义
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, n_conv=3):
        super().__init__()
        layers = []
        for i in range(n_conv):
            layers.append(nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1))
            layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)
    def forward(self, x):
        return self.block(x)

class Encoder(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, depth=4, n_conv=3):
        super().__init__()
        self.blocks = nn.ModuleList()
        self.pools = nn.ModuleList()
        ch = in_ch
        for i in range(depth):
            out_ch = base_ch * (2 ** i)
            self.blocks.append(ConvBlock(ch, out_ch, n_conv))
            self.pools.append(nn.MaxPool2d(2))
            ch = out_ch
        self.out_ch = ch
    def forward(self, x):
        skips = []
        for blk, pool in zip(self.blocks, self.pools):
            x = blk(x)
            skips.append(x)
            x = pool(x)
        return x, skips

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, n_conv=3):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        layers = [nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1), nn.ReLU(inplace=True)]
        cur = out_ch
        for _ in range(n_conv - 1):
            next_ch = max(out_ch // 2, 1)
            layers.extend([nn.Conv2d(cur, next_ch, 3, padding=1), nn.ReLU(inplace=True)])
            cur = next_ch
        self.conv = nn.Sequential(*layers)
        self.out_ch = cur
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
        return self.conv(torch.cat([x, skip], dim=1))

class UNetStage(nn.Module):
    def __init__(self, in_ch=1, base_ch=32, depth=4, n_conv=3):
        super().__init__()
        self.encoder = Encoder(in_ch, base_ch, depth, n_conv)
        mid_ch = self.encoder.out_ch * 2
        self.mid = nn.Sequential(
            nn.Conv2d(self.encoder.out_ch, mid_ch, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, self.encoder.out_ch, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.decoders = nn.ModuleList()
        cur_ch = self.encoder.out_ch
        for i in reversed(range(depth)):
            skip_ch = base_ch * (2 ** i)
            block = DecoderBlock(cur_ch, skip_ch, skip_ch, n_conv)
            self.decoders.append(block)
            cur_ch = block.out_ch
        self.out_ch = cur_ch
    def forward(self, x):
        x, skips = self.encoder(x)
        x = self.mid(x)
        for dec, skip in zip(self.decoders, reversed(skips)):
            x = dec(x, skip)
        return x

class DenoiseStage(nn.Module):
    def __init__(self, base_ch=32, depth=4, n_conv=3):
        super().__init__()
        self.encoder = UNetStage(1, base_ch, depth, n_conv)
        self.out = nn.Conv2d(self.encoder.out_ch, 1, 3, padding=1)
    def forward(self, x):
        feat = self.encoder(x)
        return F.relu(self.out(feat))

class DeconvStage(nn.Module):
    def __init__(self, base_ch=32, depth=4, n_conv=3, upsample=True):
        super().__init__()
        self.upsample = upsample
        self.stage2 = UNetStage(1, base_ch, depth, n_conv)
        self.refine1 = nn.Conv2d(self.stage2.out_ch, 128, 3, padding=1)
        self.refine2 = nn.Conv2d(128, 128, 3, padding=1)
        self.out2 = nn.Conv2d(128, 1, 3, padding=1)
    def forward(self, x):
        f2 = self.stage2(x)
        if self.upsample:
            f2 = F.interpolate(f2, scale_factor=2, mode="nearest")
        return F.relu(self.out2(F.relu(self.refine2(F.relu(self.refine1(f2))))))

class TwoStageUNet(nn.Module):
    def __init__(self, base_ch=32, depth=4, n_conv=3, upsample=True):
        super().__init__()
        self.upsample = upsample
        self.denoise = DenoiseStage(base_ch, depth, n_conv)
        self.deconv = DeconvStage(base_ch, depth, n_conv, upsample)
    def forward(self, x):
        denoised = self.denoise(x)
        deconv = self.deconv(denoised)
        return denoised, deconv

# 测试模型
model = TwoStageUNet().to(device)
x = torch.randn(1, 1, 160, 160).to(device)
denoised, deconv = model(x)
print(f"Input: {x.shape} -> Denoised: {denoised.shape}, Deconv: {deconv.shape}")

In [ ]:
# 快速训练测试（3 个 iteration）
print("开始快速训练测试...")

dataset = TwoStageDataset("datasets/Microtubule/train_data", n_samples=10)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.L1Loss()

model.train()
for i, (input_g, gt_g) in enumerate(loader):
    input_g = input_g.to(device)
    gt_g = gt_g.to(device)
    
    denoised, deconv = model(input_g)
    
    loss_denoise = criterion(denoised, gt_g)
    loss_deconv = criterion(deconv, gt_g)
    
    loss = 0.5 * loss_denoise + 0.5 * loss_deconv
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Iter {i+1}: loss={loss.item():.6f}, denoise={loss_denoise.item():.6f}, deconv={loss_deconv.item():.6f}")
    
    if i >= 2:  # 只测试 3 次
        break

print("✓ 训练测试完成！")

In [ ]:
# 测试推理
print("测试推理...")

model.eval()
with torch.no_grad():
    test_img = load_image("datasets/Microtubule/train_data/01.tif")
    test_img = cv2.resize(test_img, (128, 128))
    test_img_norm = percentile_norm(test_img)
    
    # 添加 noise 和 padding
    noisy = test_img_norm + np.random.normal(0, 0.1, test_img_norm.shape).astype(np.float32)
    noisy = np.clip(noisy, 0, 1)
    noisy_padded = np.pad(noisy, ((16, 16), (16, 16)), mode='constant')
    
    x = torch.from_numpy(noisy_padded[None, None]).float().to(device)
    denoised, deconv = model(x)
    
    denoised_np = denoised[0, 0].cpu().numpy()[16:144, 16:144]
    deconv_np = deconv[0, 0].cpu().numpy()[32:288, 32:288]  # 2x upsample
    
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 4, 1)
    plt.imshow(test_img_norm, cmap="gray")
    plt.title("Original")
    plt.axis("off")
    
    plt.subplot(1, 4, 2)
    plt.imshow(noisy, cmap="gray")
    plt.title("Noisy Input")
    plt.axis("off")
    
    plt.subplot(1, 4, 3)
    plt.imshow(percentile_norm(denoised_np), cmap="gray")
    plt.title("Denoised")
    plt.axis("off")
    
    plt.subplot(1, 4, 4)
    plt.imshow(percentile_norm(deconv_np), cmap="gray")
    plt.title("Deconvolved (2x)")
    plt.axis("off")
    
    plt.tight_layout()
    plt.savefig("quick_start_result.png", dpi=150)
    plt.show()
    
    print("✓ 推理测试完成！结果已保存到 quick_start_result.png")